# OUARTI · Poste de guidage

Choisir un mode dans la cellule suivante puis **Tout exécuter**. Le HUD fonctionne avec les seuls agents 1 et 2. Le protocole de recette est dans PROTOCOLE.md.

In [ ]:
# RÉGLAGES — un seul mode par lancement. Les versions sont affichées après entraînement.
MODE = 'inventaire'
# recette_hors_jeu, tester_a1, tester_a2, tester_a3, tester_chaine, demo_hud, annoter_train,
# tester_integration, evaluer_a2_reel, annoter_validation, entrainer_a1_reel, hud_direct, entrainer_a3, evaluer_a3
VERSION_A1 = ''
VERSION_A2 = ''
VERSION_A3 = ''  # vide pour commencer un nouveau pilote
SESSION_TRAIN = 'session_a_renseigner_train'
SESSION_VALIDATION = 'session_a_renseigner_validation'
EPOCHS_REELS = 15
CADRAGE_REEL = 'cockpit_abarth500_v1'  # ou 'historique' pour reproduire la géométrie précédente
AFFICHAGE = 'notebook'  # ou 'superposition' en mode fenêtré sans bordure
DUREE_HUD_S = 90
STAGE = 1
TRACK = 'ks_barcelona-layout_gp'
VOITURE = 'abarth500'
PAS_PPO = 4096
EPISODES = 10

In [ ]:
from pathlib import Path
import sys
import time
import torch
import numpy as np
import json

RACINE = next(p/'Code' for p in [Path.cwd(),*Path.cwd().parents] if (p/'Code'/'06_Trois_Agents').is_dir())
CARNETS = RACINE/'06_Trois_Agents'
GYM_ROOT = RACINE.parent/'assetto_corsa_gym'/'assetto_corsa_gym'
OBJECTIF_DISTANCE_M = {1:10.,2:30.,3:100.,4:None}[STAGE]
torch.set_num_threads(min(4,torch.get_num_threads()))

def charger_carnet(nom):
    fichier = CARNETS/nom
    cellules = json.loads(fichier.read_text(encoding='utf8'))['cells']
    for index,c in enumerate(cellules):
        if c['cell_type']=='code':
            exec(compile(''.join(c['source']),f"{fichier}#cellule-{index}",'exec'),globals())

MODES = {
    'recette_hors_jeu':(), 'inventaire':(), 'tester_a1':(1,), 'tester_a2':(2,), 'tester_a3':(3,),
    'annoter_train':(1,), 'annoter_validation':(1,), 'entrainer_a1_reel':(1,),
    'tester_chaine':(1,2), 'demo_hud':(1,2), 'hud_direct':(1,2),
    'evaluer_a2_reel':(1,2),'tester_integration':(1,2,3), 'entrainer_a3':(1,2,3), 'evaluer_a3':(1,2,3)}
if MODE not in MODES:
    raise ValueError(f"Mode inconnu. Choisir parmi {list(MODES)}")
for numero in MODES[MODE]:
    charger_carnet({1:'01_Perception.ipynb',2:'02_Trajectoire.ipynb',3:'03_Pilote_PPO.ipynb'}[numero])
if MODE in ('hud_direct','entrainer_a3','evaluer_a3'):
    charger_carnet('Capteurs_lecture.ipynb')
if MODE in ('entrainer_a3','evaluer_a3'):
    charger_carnet('Pilotage_Assetto.ipynb')

In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import uuid
import torch

def empreinte(path):
    h = hashlib.sha256()
    with path.open('rb') as flux:
        for bloc_lu in iter(lambda: flux.read(1<<20),b''):
            h.update(bloc_lu)
    return h.hexdigest()

def archiver(type_agent, suffixe, ecrire, details):
    ident = datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    parent = RACINE/'models'/'trois_agents'/type_agent
    temporaire = parent/('.tmp_'+ident)
    temporaire.mkdir(parents=True,exist_ok=False)
    fichier = temporaire/('agent'+suffixe)
    ecrire(fichier)
    meta = dict(details,version=ident,type_agent=type_agent,contrat=1,
                fichier=fichier.name,sha256=empreinte(fichier))
    (temporaire/'manifest.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf8')
    temporaire.rename(parent/ident)
    return ident

def relire(type_agent, ident):
    if not isinstance(ident,str) or not re.fullmatch(r'[0-9]{8}_[0-9]{6}_[a-f0-9]{8}',ident):
        raise ValueError(f"Renseigner la version exacte de {type_agent} dans RÉGLAGES.")
    dossier = RACINE/'models'/'trois_agents'/type_agent/ident
    meta = json.loads((dossier/'manifest.json').read_text(encoding='utf8'))
    suffixe = '.zip' if type_agent=='pilote' else '.pt'
    fichier = dossier/('agent'+suffixe)
    if (meta['contrat']!=1 or meta['type_agent']!=type_agent or meta['version']!=ident
            or meta['fichier']!=fichier.name or empreinte(fichier)!=meta['sha256']):
        raise ValueError(f"Archive incompatible ou altérée : {dossier}")
    return fichier,meta

def charger_reseau(type_agent, ident, classe):
    fichier,meta = relire(type_agent,ident)
    modele = classe()
    modele.load_state_dict(torch.load(fichier,map_location='cpu',weights_only=True))
    return modele.eval(),meta

def rapport(nom, donnees):
    dossier = RACINE/'reports'/'trois_agents'
    dossier.mkdir(exist_ok=True,parents=True)
    path = dossier/(nom+'_'+uuid.uuid4().hex[:10]+'.json')
    path.write_text(json.dumps(donnees,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf8')
    print(donnees, '\nRapport :',path)
    return path

In [ ]:
def preparer_a1(reel):
    rng = np.random.default_rng(72)
    paires = [scene_synthetique(rng) for _ in range(192)]
    modele = entrainer_perception(paires,32)
    mesures = {}
    if reel:
        apprentissage,validation = dossiers_annotations()
        train,val = paires_reelles(apprentissage),paires_reelles(validation)
        import hashlib
        h_train = {hashlib.sha256(im.tobytes()).digest() for im,_ in train}
        if any(hashlib.sha256(im.tobytes()).digest() in h_train for im,_ in val):
            raise ValueError("Image commune aux sessions d’apprentissage et de validation")
        if any(not ((m!=255)&(m!=PISTE)).any() for _,m in val):
            raise ValueError("Annoter aussi le hors-piste dans chaque image de validation")
        modele = entrainer_perception(train+paires[:min(len(train),32)],EPOCHS_REELS,modele=modele)
        mesures['iou_validation_reelle'] = mesurer_perception(modele,val)
        mesures.update(images_train=len(train),images_validation=len(val),
                       session_train=SESSION_TRAIN,session_validation=SESSION_VALIDATION)
    if not reel:
        mesures.update(tester_a1(modele))
    ident = archiver('perception','.pt',lambda f: torch.save(modele.state_dict(),f),
                     dict(mesures,domaine='captures' if reel else 'synthetique',classes=CLASSES))
    return dict(mesures,version_a1=ident)

def preparer_a2():
    modele = entrainer_planificateur()
    mesures = tester_a2(modele)
    ident = archiver('trajectoire','.pt',lambda f: torch.save(modele.state_dict(),f),mesures)
    return dict(mesures,version_a2=ident)

def reseaux_guidage(reel, *, hors_jeu=False):
    a1,meta = charger_reseau('perception',VERSION_A1,PerceptionUNet)
    if reel and (meta.get('domaine')!='captures' or (not hors_jeu and meta.get('iou_validation_reelle',{}).get('piste',0)<.75)):
        raise ValueError("Valider A1 sur une autre session de piste annotée (IoU piste minimale 0,75)")
    a2,_ = charger_reseau('trajectoire',VERSION_A2,Planificateur)
    if reel:
        if CADRAGE_REEL not in (COCKPIT_ABARTH, "historique"):
            raise ValueError(f"Cadrage inconnu : {CADRAGE_REEL}")
        a2.cadrage = CADRAGE_REEL
    return a1,a2

def dossiers_annotations():
    noms = [SESSION_TRAIN,SESSION_VALIDATION]
    if len(set(noms))!=2 or any(not re.fullmatch(r'session_[A-Za-z0-9_-]+',n) for n in noms):
        raise ValueError("Choisir deux sessions de capture distinctes")
    return [RACINE/'datasets'/'segmentation'/n for n in noms]

def enregistrer_pilote(modele):
    meta = dict(domaine='assetto',cadrage=CADRAGE_REEL,perception=VERSION_A1,trajectoire=VERSION_A2,parent=VERSION_A3,
                track=TRACK,voiture=VOITURE,pas=int(modele.num_timesteps),stage=STAGE,
                objectif_m=OBJECTIF_DISTANCE_M)
    ident = archiver('pilote','.zip',modele.save,meta)
    print('Version A3',ident,'à',modele.num_timesteps,'pas')
    return ident

def lancer_pilote(evaluation):
    a1,a2 = reseaux_guidage(True)
    if evaluation and not VERSION_A3:
        raise ValueError("Choisir une version du pilote pour évaluer")
    precedent = None
    if VERSION_A3:
        precedent,meta = relire('pilote',VERSION_A3)
        if meta.get('domaine')!='banc' and meta.get('cadrage','historique')!=CADRAGE_REEL:
            raise ValueError('Choisir le cadrage associé à la version du pilote')
        if meta.get('domaine')=='banc' and evaluation:
            raise ValueError('Le pilote de banc doit être entraîné dans Assetto avant la recette sur piste')
        if meta.get('domaine')!='banc' and (meta['perception'],meta['trajectoire'])!=(VERSION_A1,VERSION_A2):
            raise ValueError("Reprendre les versions A1/A2 associées au pilote")
    if STAGE not in (1,2,3,4) or PAS_PPO<=0 or EPISODES<=0:
        raise ValueError("Vérifier STAGE, PAS_PPO et EPISODES")
    a1.observer(np.zeros((96,128),np.uint8))
    with torch.inference_mode():
        a2(torch.zeros(1,7,2),torch.zeros(1,2))
    core = LiveAssettoVisionEnv(TRACK,VOITURE,episode_seconds={1:30,2:60,3:120,4:900}[STAGE],
                                curriculum_stage=STAGE)
    env = SuiviTrajectoire(core,a1,a2)
    try:
        taille_lot = {1:256,2:512,3:1024,4:8192}[STAGE]
        if precedent:
            modele = PPO.load(precedent,env=env,device='cpu',n_steps=taille_lot)
        else:
            modele = nouveau_pilote(env,n_steps=taille_lot)
        modele.predict(np.zeros(10,np.float32),deterministic=True)
        if not evaluation:
            modele.learn(total_timesteps=PAS_PPO,reset_num_timesteps=not bool(precedent),
                         callback=FreinPendantMiseAJour(core,enregistrer_pilote))
            return {'version_a3':enregistrer_pilote(modele),'pas':int(modele.num_timesteps)}
        lignes = []
        for i in range(EPISODES):
            obs,_ = env.reset()
            erreurs = []
            while True:
                action,_ = modele.predict(obs,deterministic=True)
                obs,_,term,trunc,info = env.step(action)
                if 'ecart_image_normalise' in info:
                    erreurs.append(info['ecart_image_normalise'])
                if term or trunc:
                    ligne = dict(episode=i+1,arrivee=bool(info['milestone_reached'] or info['lap_finished']),
                                 tour=bool(info['lap_finished']),sortie=bool(info['offtrack']),
                                 collision=bool(info['collision']),perte_guidage=info.get('guidage_perdu'),
                                 progression_m=info['episode_progress_m'],
                                 ecart_image_moyen=float(np.mean(erreurs)) if erreurs else None)
                    lignes.append(ligne)
                    rapport('episode_a3',dict(ligne,versions=[VERSION_A1,VERSION_A2,VERSION_A3]))
                    break
        return {'episodes':lignes,'version_a3':VERSION_A3}
    finally:
        env.close()

In [ ]:
def verifier_ligne(a2, probabilite, masque, raw):
    try:
        plan = a2.proposer(probabilite, raw['speed'], raw['angular_velocity_y'])
    except GuidageIndisponible as exc:
        return dict(etat='indisponible', raison=str(exc))
    h,w = masque.shape
    trace = np.zeros_like(masque)
    pts = np.rint(plan.xy*[w-1,h-1]).astype(np.int32)
    cv2.polylines(trace,[pts],False,1,2)
    labels = masque[trace!=0]
    if ((labels!=PISTE)&(labels!=255)).any():
        etat = 'hors_masque'
    elif (labels==255).any():
        etat = 'annotation_incomplete'
    else:
        etat = 'dans_piste'
    return dict(etat=etat, cible_kmh=plan.cible_kmh, points=plan.xy.tolist())

def evaluer_a2_reel(a1,a2):
    dossier = dossiers_annotations()[1]
    session = RACINE/'datasets'/SESSION_VALIDATION
    index = {r['image']:r for r in (json.loads(s) for s in (session/'frames.jsonl').read_text(encoding='utf8').splitlines())}
    telemetrie = {int(r['steps']):r for r in (json.loads(s) for s in (session/'telemetry.jsonl').read_text(encoding='utf8').splitlines())}
    fichiers = sorted((dossier/'images').glob('*.png'))
    if not fichiers:
        raise ValueError(f"Aucune capture annotée dans {dossier}")
    lignes = []
    for f in fichiers:
        image = cv2.imread(str(f),cv2.IMREAD_GRAYSCALE)
        masque = cv2.imread(str(dossier/'masques'/f.name),cv2.IMREAD_UNCHANGED)
        if image is None or masque is None or masque.shape!=image.shape or not np.isin(masque,[*range(len(CLASSES)),255]).all():
            raise ValueError(f"Image ou masque indexé invalide : {f.name}")
        fr = index['images/'+f.name]
        raw = telemetrie[int(fr['steps'])]
        ecart = abs(float(fr['camera_gap_s']))
        if not np.isfinite(ecart) or ecart>.4:
            lignes.append(dict(image=f.name,etat='desynchronise',reference={'etat':'desynchronise'}))
            continue
        # Les pixels ignorés (255) ne désignent jamais un passage praticable.
        reference = verifier_ligne(a2,(masque==PISTE).astype(np.float32),masque,raw)
        _,p = a1.observer(image)
        mesure = verifier_ligne(a2,p,masque,raw)
        lignes.append(dict(image=f.name,vitesse_ms=raw['speed'],reference=reference,**mesure))
    def bilan(resultats):
        return dict(couverture=sum(r['etat']=='dans_piste' for r in resultats)/len(resultats),
                    franchissements=sum(r['etat']=='hors_masque' for r in resultats))
    return dict(session=SESSION_VALIDATION,images=len(lignes),cadrage=a2.cadrage,
                reference=bilan([r['reference'] for r in lignes]),**bilan(lignes),details=lignes)


In [ ]:
def recette_hors_jeu():
    global VERSION_A1,VERSION_A2,VERSION_A3
    charger_carnet('01_Perception.ipynb')
    a = preparer_a1(False)
    VERSION_A1 = a['version_a1']
    rapport('tester_a1',a)
    charger_carnet('02_Trajectoire.ipynb')
    b = preparer_a2()
    VERSION_A2 = b['version_a2']
    rapport('tester_a2',b)
    a1,a2 = reseaux_guidage(False)
    guidage = tester_chaine_humaine(a1,a2)
    rapport('chaine_humaine',dict(guidage,versions=[VERSION_A1,VERSION_A2]))
    charger_carnet('03_Pilote_PPO.ipynb')
    c = tester_a3()
    VERSION_A3 = c['version_a3']
    rapport('tester_a3',c)
    chaine = tester_integration(a1,a2)
    return dict(perception=a,trajectoire=b,guidage_humain=guidage,pilote=c,integration=chaine)

In [ ]:
def executer():
    if MODE=='recette_hors_jeu':
        return recette_hors_jeu()
    if MODE=='inventaire':
        for type_agent in ('perception','trajectoire','pilote'):
            noms = [p.parent.name for p in sorted((RACINE/'models'/'trois_agents'/type_agent).glob('*/manifest.json')) if not p.parent.name.startswith('.')]
            print(type_agent, noms)
        return
    if MODE in ('annoter_train','annoter_validation'):
        i = int(MODE=='annoter_validation')
        session = (SESSION_TRAIN,SESSION_VALIDATION)[i]
        annoter_session(RACINE/'datasets'/session,dossiers_annotations()[i])
        return
    if MODE in ('tester_a1','entrainer_a1_reel'):
        return preparer_a1(MODE=='entrainer_a1_reel')
    if MODE=='tester_a2':
        return preparer_a2()
    if MODE=='tester_integration':
        return tester_integration(*reseaux_guidage(False))
    if MODE=='evaluer_a2_reel':
        return evaluer_a2_reel(*reseaux_guidage(True, hors_jeu=True))
    if MODE=='tester_a3':
        return tester_a3()
    if MODE in ('entrainer_a3','evaluer_a3'):
        print("Basculer sur le jeu dans les 5 secondes.")
        time.sleep(5)
        return lancer_pilote(MODE=='evaluer_a3')
    a1,a2 = reseaux_guidage(MODE=='hud_direct')
    if MODE=='hud_direct':
        return afficher_hud(a1,a2,DUREE_HUD_S)
    if MODE=='demo_hud':
        from IPython.display import display, Image
        rng = np.random.default_rng(564)
        for _ in range(40):
            image,_ = scene_synthetique(rng)
            try:
                _,p = a1.observer(image)
                plan = a2.proposer(p,6.,0.)
            except GuidageIndisponible:
                continue
            ok,jpg = cv2.imencode('.jpg',dessiner(image,plan))
            if ok:
                display(Image(data=jpg.tobytes()))
                return {'domaine':'synthétique','cible_kmh':plan.cible_kmh,'points':plan.xy.tolist()}
        raise ValueError("Aucune proposition sur ces scènes")
    return tester_chaine_humaine(a1,a2)

resultat = executer()
if resultat is not None:
    rapport(MODE,dict(resultat,selections=[VERSION_A1,VERSION_A2,VERSION_A3]))